# Character BiLSTM baseline (MODEL 2)

This notebook follows the URLBERT training/evaluation contract and the character CNN preprocessing: shared splits, shared character vocabulary, phishing label 0, validation checkpointing, and comparable prediction files.

In [ ]:
# Run once in the project environment. PyTorch is supplied by the environment.
%pip install -q -r ../requirements.txt

In [ ]:
import json
import os
import re
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

sys.path.append('../src')
from evaluate import evaluate, path_baseline, save_preds
from load_data import load_splits

## Configuration

In [ ]:
SMOKE = False
OVERFIT = False
SEED = 42
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 100 if OVERFIT else 1 if SMOKE else 20
EVAL_EVERY = 10 if SMOKE or OVERFIT else 500
PATIENCE = 3
MIN_DELTA = 1e-4

TAG = 'bilstm_overfit' if OVERFIT else 'bilstm_smoke' if SMOKE else 'bilstm'
RESULTS = '../results'
CKPT = f'../models/{TAG}_model2.pt'
LOG = f'{RESULTS}/{TAG}_train_log.csv'
os.makedirs(RESULTS, exist_ok=True)
os.makedirs('../checkpoints', exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
use_amp = device.type == 'cuda'
print('device:', device, '| torch:', torch.__version__)

## Load data and reproduce the character-CNN preprocessing

The shared vocabulary has 67 entries: `<PAD>`, `<UNK>`, and 65 retained URL characters. The shared maximum length is 256. Canonicalization is intentionally the same as `cnn_baseline.ipynb`; do not compare scores across models if one model receives canonicalized URLs and another receives raw URLs.

In [ ]:
splits = load_splits()

def canonicalize_model_url(url):
    url = str(url).strip()
    url = re.sub(r'^[A-Za-z][A-Za-z0-9+.\-]*://', '', url)
    url = re.sub(r'^www\d*\.', '', url, flags=re.IGNORECASE)
    return re.sub(r'/+$', '', url)

for split in splits:
    splits[split] = splits[split].copy()
    splits[split]['URL'] = splits[split]['URL'].map(canonicalize_model_url)

if SMOKE:
    splits = {s: d.sample(2000 if s == 'train' else 500, random_state=SEED) for s, d in splits.items()}
if OVERFIT:
    splits['train'] = splits['val'] = splits['train'].groupby('label').head(3)

with open('../data/vocab.json') as f:
    vocab = json.load(f)
with open('../data/config.json') as f:
    config = json.load(f)

char_to_idx = {c: i for i, c in enumerate(vocab['index_to_string'])}
PAD_IDX = vocab['pad_idx']
UNK_IDX = vocab['unk_idx']
MAX_LEN = config['max_len']
print({s: len(d) for s, d in splits.items()})
print(f'vocab={len(char_to_idx)}, MAX_LEN={MAX_LEN}, PAD_IDX={PAD_IDX}, UNK_IDX={UNK_IDX}')

In [ ]:
class URLDataset(Dataset):
    def __init__(self, df):
        self.urls = df['URL'].tolist()
        self.labels = df['label'].astype('int64').tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        ids = [char_to_idx.get(c, UNK_IDX) for c in self.urls[index][:MAX_LEN]]
        return torch.tensor(ids, dtype=torch.long), self.labels[index]

def collate(batch):
    ids, labels = zip(*batch)
    lengths = torch.tensor([len(x) for x in ids], dtype=torch.long)
    ids = pad_sequence(ids, batch_first=True, padding_value=PAD_IDX)
    return ids, lengths, torch.tensor(labels, dtype=torch.long)

datasets = {s: URLDataset(df) for s, df in splits.items()}
train_loader = DataLoader(datasets['train'], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(datasets['val'], batch_size=512, shuffle=False, collate_fn=collate)
test_loader = DataLoader(datasets['test'], batch_size=512, shuffle=False, collate_fn=collate)
ids, lengths, labels = next(iter(train_loader))
print('input:', tuple(ids.shape), '| lengths:', tuple(lengths.shape), '| labels:', tuple(labels.shape))

## BiLSTM model

The embedding maps characters to vectors. Packing prevents padding from being processed. The final forward and backward hidden states are concatenated, then classified with two logits. Since the dataset labels are unchanged, class 0 is phishing and `softmax(logits)[:, 0]` is `p_phish`.

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 2)

    def forward(self, ids, lengths):
        embedded = self.embedding(ids)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        representation = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(self.dropout(representation))

model = BiLSTMClassifier(len(char_to_idx)).to(device)
with torch.no_grad():
    print('model output:', tuple(model(ids.to(device), lengths.to(device)).shape))
print(f'parameters: {sum(p.numel() for p in model.parameters()):,}')

## Evaluation helpers

In [ ]:
loss_fn = nn.CrossEntropyLoss()

@torch.no_grad()
def run_eval(loader):
    model.eval()
    total_loss, probabilities = 0.0, []
    for ids, lengths, labels in loader:
        ids, lengths, labels = ids.to(device), lengths.to(device), labels.to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            logits = model(ids, lengths).float()
        total_loss += loss_fn(logits, labels).item() * len(labels)
        probabilities.append(torch.softmax(logits, dim=-1)[:, 0].cpu())
    return total_loss / len(loader.dataset), torch.cat(probabilities).numpy()

def predict(urls):
    frame = pd.DataFrame({'URL': [canonicalize_model_url(u) for u in urls], 'label': 1})
    dataset = URLDataset(frame)
    return run_eval(DataLoader(dataset, batch_size=512, collate_fn=collate))[1]

## Training

This uses AdamW, six-percent linear warmup/decay, gradient clipping, validation loss checkpointing, and early stopping. Validation also runs at the end of every epoch, so smoke and overfit modes always produce a checkpoint.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = MAX_EPOCHS * len(train_loader)
warmup = int(0.06 * total_steps)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda step: step / max(1, warmup) if step < warmup else max(0.0, (total_steps - step) / max(1, total_steps - warmup)),
)
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
writer = SummaryWriter(f'../runs/{TAG}')
log, running = [], []
best_val, bad, step, stopped = float('inf'), 0, 0, False

def validate_and_checkpoint():
    global best_val, bad
    val_loss, p_val = run_eval(val_loader)
    metrics = evaluate(datasets['val'].labels, p_val)
    improved = val_loss < best_val - MIN_DELTA
    if improved:
        best_val, bad = val_loss, 0
        torch.save(model.state_dict(), CKPT)
    else:
        bad += 1
    return val_loss, metrics, improved

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    for ids, lengths, labels in tqdm(train_loader, desc=f'epoch {epoch}'):
        ids, lengths, labels = ids.to(device), lengths.to(device), labels.to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            loss = loss_fn(model(ids, lengths), labels)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        step += 1
        running.append(loss.item())
        writer.add_scalar('loss/train_step', loss.item(), step)

        if step % EVAL_EVERY == 0:
            val_loss, metrics, improved = validate_and_checkpoint()
            row = {'step': step, 'epoch': epoch, 'train_loss': np.mean(running), 'val_loss': val_loss,
                   'val_acc': metrics['accuracy'], 'val_recall': metrics['recall'], 'val_f1': metrics['f1'],
                   'lr': scheduler.get_last_lr()[0], 'best': improved}
            log.append(row)
            running = []
            pd.DataFrame(log).to_csv(LOG, index=False)
            print(f"step {step} | train {row['train_loss']:.4f} | val {val_loss:.4f} | f1 {metrics['f1']:.4f}" + (' | best' if improved else ''))
            if bad >= PATIENCE:
                print(f'early stopping: val loss did not improve for {PATIENCE} evals')
                stopped = True
                break
    if stopped:
        break
    if step % EVAL_EVERY != 0:
        val_loss, metrics, improved = validate_and_checkpoint()
        print(f"epoch {epoch} | val {val_loss:.4f} | f1 {metrics['f1']:.4f}" + (' | best' if improved else ''))
        if bad >= PATIENCE:
            print(f'early stopping: val loss did not improve for {PATIENCE} evals')
            break
writer.close()

## Test evaluation and comparable outputs

In [ ]:
if not os.path.exists(CKPT):
    raise FileNotFoundError(f'No checkpoint found at {CKPT}; run the training cell first.')
model.load_state_dict(torch.load(CKPT, map_location=device, weights_only=True))
test = splits['test']
test_loss, p_test = run_eval(test_loader)
test_metrics = evaluate(test['label'], p_test)
save_preds(f'{RESULTS}/{TAG}_test_original.csv', test['URL'], test['label'], p_test)
with open(f'{RESULTS}/{TAG}_test_original.json', 'w') as f:
    json.dump(test_metrics, f, indent=1)

comparison = pd.DataFrame({
    'BiLSTM': test_metrics,
    'path rule': evaluate(test['label'], path_baseline(test['URL'])),
}).drop('confusion_matrix').T
display(comparison.astype(float).round(4))
print('confusion matrix [true phishing, true legitimate]:')
print(np.asarray(test_metrics['confusion_matrix']))